# Cortex Phantom Core 2 — SFT + DPO tanulókör (Kaggle, ingyenes T4)

**Mi az új a v2-ben (az 1. körhöz képest):**
- a tanítókorpusz a szerveren készül (`tools/phantom_mix.py`): FineWeb-stílusú minőségi
  szűrés, dedup, és **valódi RLAIF preferencia-párok** az éjjeli tanítógyárból;
- a nagy fájlok az R2-ben vannak (`cortexai/phantom/mix/`), itt csak letöltjük őket;
- **DPO futam** a SFT után: a rejected válaszok az alkotmány által tiltott hibákkal
  szennyezve (töltelék, ismétlés, angol keverés) — a modell pont azt tanulja elutasítani,
  amit a konstíció tilt (Anthropic Constitutional AI + DPO recept).


In [ ]:
# 1) GPU és környezet
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nincs!")
assert torch.cuda.is_available(), "GPU runtime kell (Settings → Accelerator → GPU T4)"
!nvidia-smi | head -6

In [ ]:
# 2) csomagok (2-3 perc) — a trl a DPO-hoz kell
!pip install -q unsloth peft transformers datasets accelerate bitsandbytes trl==0.11.*
import trl, unsloth
print("trl", trl.__version__)

In [ ]:
# 3) a repó behúzása (nyilvános, nem kell token)
import os
if not os.path.exists('/kaggle/working/cortexai-ios'):
    !git clone --depth 1 https://github.com/Csakegymacska/cortexai-ios.git /kaggle/working/cortexai-ios
WORK = '/kaggle/working/cortexai-ios'
!ls {WORK}/tools | head

In [ ]:
# 4) tanítókorpusz-v2 letöltése az R2-ből (cortexai/phantom/mix/)
#    A mixet a szerveren a tools/phantom_mix.py gyártja: minőségi szűrés + dedup
#    + DPO preferenciák (valódi RLAIF + konstíció-tiltott szintetikus hibák).
#    A CF titkok a Kaggle Secrets-ből jönnek (CLOUDFLARE_EMAIL / CLOUDFLARE_API_KEY /
#    CF_ACCOUNT_ID). Ha nincsenek meg, a cella hibát jelez.
import json, os, urllib.request
from pathlib import Path

env = {}
for k in ("CLOUDFLARE_EMAIL", "CLOUDFLARE_API_KEY", "CF_ACCOUNT_ID"):
    env[k] = os.environ.get(k) or (open(f"/kaggle/input/{k.lower()}/{k}").read().strip() if os.path.exists(f"/kaggle/input/{k.lower()}/{k}") else None)
assert all(env.values()), "Hiányzó CF titkok a Kaggle Secrets-ből!"

def r2_get(key):
    url = f"https://api.cloudflare.com/client/v4/accounts/{env['CF_ACCOUNT_ID']}/r2/buckets/cortex-app/objects/{key}"
    req = urllib.request.Request(url)
    req.add_header("X-Auth-Email", env["CLOUDFLARE_EMAIL"])
    req.add_header("X-Auth-Key", env["CLOUDFLARE_API_KEY"])
    return urllib.request.urlopen(req, timeout=120).read()

MIX = Path("/var/tmp/phantom-mix"); MIX.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "val.jsonl", "dpo.jsonl", "manifest.json"):
    body = r2_get(f"cortexai/phantom/mix/{name}")
    (MIX / name).write_bytes(body)
    print(f"{name}: {len(body)} B")
print((MIX / "manifest.json").read_text()[:400])

In [ ]:
# 5) SFT — 4 bites QLoRA, 1,5B bázis
#    Ha a loss 1,2 alá megy és nem csökken tovább: kész. Ha 2,5 fölött áll: kevés lépés.
BASE  = "Qwen/Qwen2.5-1.5B-Instruct"   # 16 GB T4-re kényelmes; nagyobb: Qwen2.5-3B-Instruct
STEPS = 1500                            # ~2-2,5 óra T4-en
OUT   = "/kaggle/working/phantom"
!cd /kaggle/working/cortexai-ios && python3 cortex-phantom/train_gpu.py \
    --data /var/tmp/phantom-mix --base {BASE} --steps {STEPS} --out {OUT}

In [ ]:
# 6) DPO futam — a konstíció preferenciáin tanul
#    A dpo.jsonl sorai: {prompt, chosen, rejected}. A rejected az alkotmány TILTOTT
#    hibáival szennyezett változat (töltelékszavak, ismétlés, angol keverés, kitérés) —
#    a modell megtanulja ezeket elutasítani. RLAIF élőben: az éjjeli gyár valódi
#    lektorált preferenciái is benne vannak.
import json
from datasets import Dataset
from trl import DPOConfig, DPOTrainer
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

rows = [json.loads(x) for x in open("/var/tmp/phantom-mix/dpo.jsonl", encoding="utf-8") if x.strip()]
print("DPO párok:", len(rows))
ds = Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]} for r in rows])

tok = AutoTokenizer.from_pretrained(OUT)
model = AutoModelForCausalLM.from_pretrained(
    OUT, torch_dtype=torch.float16, device_map="auto", load_in_4bit=True)

cfg = DPOConfig(
    output_dir="/kaggle/working/phantom-dpo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6,          # DPO-hoz kisebb LR, mint az SFT-hez
    max_length=1024,
    beta=0.1,                    # a preferencia-erő (0.05-0.3 között szokás)
    logging_steps=10,
    save_strategy="epoch",
    bf16=False, fp16=True,
)
trainer = DPOTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tok)
trainer.train()
model.save_pretrained("/kaggle/working/phantom-dpo/final")
tok.save_pretrained("/kaggle/working/phantom-dpo/final")
print("DPO kész → /kaggle/working/phantom-dpo/final")

In [ ]:
# 7) mérés: ugyanaz a mérőpad, mint a napi körben (tools/phantom_eval.py)
#    Kaggle-en a válaszokat itt, helyben kérjük le a tanított modelltől; a pontozást
#    a szerveren a megszokott xKiro-bíró adja. Ha nincs meg a kulcs, a futam kimarad.
!cd /kaggle/working/cortexai-ios && python3 cortex-phantom/brain.py --eval-local /kaggle/working/phantom-dpo/final 2>/dev/null || echo "helyi eval: a 8. lépés után, a szerveren

In [ ]:
# 8) csomagolás (a LoRA adapter pár 10 MB) — ezt visszük a szerverre
#    A phantom-dpo/final adapter letölthető Kaggle Outputként; a szerveren a
#    következő tanítási kör betölti (CORTEX-TANITAS.md szerint).
!cd /kaggle/working/phantom-dpo/final && ls -la | head
!du -sh /kaggle/working/phantom-dpo/final